In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# التغذية الراجعة الكلاسيكية وتدفق التحكم (الدوائر الديناميكية)

<Accordion>
<AccordionItem title="إصدارات الحزم">

تم تطوير الكود في هذه الصفحة باستخدام المتطلبات التالية.
ننصح باستخدام هذه الإصدارات أو ما هو أحدث منها.

```
qiskit[all]~=2.4.0
```
</AccordionItem>
</Accordion>
الدوائر الديناميكية أدوات قوية تتيح لك قياس الـ qubits في منتصف تنفيذ دائرة كمومية ثم إجراء عمليات منطقية كلاسيكية داخل الدائرة بناءً على نتائج قياسات منتصف الدائرة. تُعرف هذه العملية أيضاً بـ _التغذية الراجعة الكلاسيكية_. وعلى الرغم من أننا في بداية فهم كيفية الاستفادة القصوى من الدوائر الديناميكية، فقد رصد مجتمع البحث الكمي عدداً من حالات الاستخدام، منها:

* التحضير الكفء لحالات الكم، كـ [حالة GHZ](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339)، [وحالة W](https://arxiv.org/abs/2403.07604) (لمزيد من المعلومات حول حالة W، راجع أيضاً ["تحضير الحالة بدوائر ضحلة باستخدام التغذية الراجعة"](https://arxiv.org/abs/2307.14840))، وفئة واسعة من [حالات حاصل الضرب المصفوفي](https://arxiv.org/abs/2404.16083)
* [التشابك الكفء بعيد المدى](https://journals.aps.org/prxquantum/abstract/10.1103/PRXQuantum.5.030339) بين الـ qubits على الشريحة ذاتها باستخدام دوائر ضحلة
* [أخذ عينات كفء من دوائر شبيهة بـ IQP](https://arxiv.org/pdf/2505.04705)

يدعم Qiskit أربعة تراكيب لتدفق التحكم للتغذية الراجعة الكلاسيكية، كل منها مُطبَّق كميثود على [`QuantumCircuit`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit). هذه التراكيب وميثوداتها المقابلة هي:

- جملة if - [`QuantumCircuit.if_test`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#if_test)
- جملة switch - [`QuantumCircuit.switch`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#switch)
- حلقة for - [`QuantumCircuit.for_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#for_loop)
- حلقة while - [`QuantumCircuit.while_loop`](https://docs.quantum.ibm.com/api/qiskit/qiskit.circuit.QuantumCircuit#while_loop)

تُعيد كل من هذه الميثودات [مدير سياق](https://docs.python.org/3/reference/datamodel.html#with-statement-context-managers) وتُستخدم عادةً في جملة `with`. يشرح باقي هذا الدليل كل من هذه التراكيب وكيفية استخدامها.

> **Caution:** ثمة بعض القيود على عمليات التغذية الراجعة الكلاسيكية وتدفق التحكم على الأجهزة الكمومية قد تؤثر على برنامجك. لمزيد من المعلومات، راجع [تنفيذ الدوائر الديناميكية](/guides/execute-dynamic-circuits).

## جملة `if`
تُستخدم جملة `if` لتنفيذ عمليات بشكل شرطي بناءً على قيمة بت كلاسيكي أو سجل كلاسيكي.

في المثال التالي، نطبّق بوابة هادامار على qubit ونقيسه. إذا كانت النتيجة 1، نطبّق بوابة X على الـ qubit مما يعيده إلى الحالة 0. ثم نقيس الـ qubit مرة أخرى. يجب أن تكون نتيجة القياس 0 باحتمالية 100%.

In [1]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister

qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)):
    circuit.x(q0)
circuit.measure(q0, c0)
circuit.draw("mpl")

# example output counts: {'0': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/101aaa8f-7061-4924-9b50-806d7e1ab728-0.avif)

يمكن إعطاء جملة `with` هدف إسناد يكون بحد ذاته مدير سياق يمكن تخزينه واستخدامه لاحقاً لإنشاء كتلة else، التي تُنفَّذ في كل مرة لا تُنفَّذ فيها محتويات كتلة `if`.

في المثال التالي، نُهيئ سجلات تحتوي على Qubitين وبتّين كلاسيكيين. نطبّق بوابة هادامار على الـ qubit الأول ونقيسه. إذا كانت النتيجة 1، نطبّق بوابة هادامار على الـ qubit الثاني؛ وإلا نطبّق بوابة X عليه. وأخيراً، نقيس الـ qubit الثاني أيضاً.

In [2]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1) = qubits
(c0, c1) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.if_test((c0, 1)) as else_:
    circuit.h(q1)
with else_:
    circuit.x(q1)
circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 260, '11': 272, '10': 492}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/1f6737fe-bc45-4d0c-b7b4-1096e2d7e14a-0.avif)

بالإضافة إلى الاشتراط على بت كلاسيكي واحد، يمكن أيضاً الاشتراط على قيمة سجل كلاسيكي مؤلف من عدة بتات.

في المثال التالي، نطبّق بوابات هادامار على Qubitين ونقيسهما. إذا كانت النتيجة `01`، أي أن الـ qubit الأول هو 1 والـ qubit الثاني هو 0، نطبّق بوابة X على qubit ثالث. وأخيراً، نقيس الـ qubit الثالث. لاحظ أننا اخترنا لوضوح الكود تحديد حالة البت الكلاسيكي الثالث وهي 0 في شرط `if`. في رسم الدائرة، يُشار إلى الشرط بالدوائر على البتات الكلاسيكية المشروط عليها. تُشير الدائرة المصمتة إلى الاشتراط على 1، بينما تُشير الدائرة المفرغة إلى الاشتراط على 0.

In [3]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.if_test((clbits, 0b001)):
    circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 269, '011': 260, '000': 252, '010': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/37ec3fa6-04b5-4165-b8d2-bae5fd238331-0.avif)

## جملة Switch
تُستخدم جملة switch لاختيار الإجراءات بناءً على قيمة بت كلاسيكي أو سجل كلاسيكي. وهي مشابهة لجملة if، غير أنه يمكنك تحديد حالات أكثر لمنطق التفرع. يطبّق المثال التالي بوابة هادامار على qubit ويقيسه. إذا كانت النتيجة 0، يُطبَّق بوابة X على الـ qubit، وإذا كانت النتيجة 1، يُطبَّق بوابة Z. يجب أن تكون نتيجة القياس 1 باحتمالية 100%.

In [4]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

circuit.h(q0)
circuit.measure(q0, c0)
with circuit.switch(c0) as case:
    with case(0):
        circuit.x(q0)
    with case(1):
        circuit.z(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/fc2bc3c3-eab1-41f0-b696-5e8b30155d55-0.avif)

نظراً لاستخدام المثال أعلاه بتاً كلاسيكياً واحداً فحسب، لم يكن ثمة إلا حالتان محتملتان، وكان بالإمكان تحقيق النتيجة ذاتها باستخدام جملة if-else. تكون حالة switch مفيدة بشكل رئيسي عند التفرع بناءً على قيمة سجل كلاسيكي مؤلف من عدة بتات. يوضح المثال التالي كيفية إنشاء حالة افتراضية (default)، التي تُنفَّذ إذا لم تتطابق أي من الحالات السابقة. لاحظ أنه في جملة switch، لا يُنفَّذ سوى كتلة واحدة فحسب. لا يوجد تسلسل تنفيذي (fallthrough).

يطبّق المثال التالي بوابات هادامار على Qubitين ويقيسهما. إذا كانت النتيجة 00 أو 11، يُطبَّق بوابة Z على الـ qubit الثالث. إذا كانت النتيجة 01، يُطبَّق بوابة Y. وإذا لم تتطابق أي من الحالات السابقة، يُطبَّق بوابة X. وأخيراً، يُقاس الـ qubit الثالث.

In [5]:
qubits = QuantumRegister(3)
clbits = ClassicalRegister(3)
circuit = QuantumCircuit(qubits, clbits)
(q0, q1, q2) = qubits
(c0, c1, c2) = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.switch(clbits) as case:
    with case(0b000, 0b011):
        circuit.z(q2)
    with case(0b001):
        circuit.y(q2)
    with case(case.DEFAULT):
        circuit.x(q2)
circuit.measure(q2, c2)

circuit.draw("mpl")

# example output counts: {'101': 267, '110': 249, '011': 265, '000': 243}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/a5d43b4c-c538-4f34-8cf3-92c2c0d26fdd-0.avif)

## حلقة For
تُستخدم حلقة for للتكرار عبر تسلسل من القيم الكلاسيكية وتنفيذ بعض العمليات في كل تكرار.

يستخدم المثال التالي حلقة for لتطبيق 5 بوابات X على qubit ثم يقيسه. ونظراً لتنفيذ عدد فردي من بوابات X، يكون التأثير الإجمالي هو قلب الـ qubit من الحالة 0 إلى الحالة 1.

In [6]:
qubits = QuantumRegister(1)
clbits = ClassicalRegister(1)
circuit = QuantumCircuit(qubits, clbits)
(q0,) = qubits
(c0,) = clbits

with circuit.for_loop(range(5)) as _:
    circuit.x(q0)
circuit.measure(q0, c0)

circuit.draw("mpl")

# example output counts: {'1': 1024}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/53a26ce5-3564-47a0-8803-c9c46db86923-0.avif)

## حلقة While
تُستخدم حلقة while لتكرار التعليمات طالما تحقق شرط معين.

يطبّق المثال التالي بوابات هادامار على Qubitين ويقيسهما. ثم ينشئ حلقة while تكرر هذا الإجراء ما دامت نتيجة القياس 11. ونتيجةً لذلك، يجب ألا تكون نتيجة القياس النهائية 11 أبداً، مع ظهور الاحتمالات المتبقية بتكرار متقارب.

In [7]:
qubits = QuantumRegister(2)
clbits = ClassicalRegister(2)
circuit = QuantumCircuit(qubits, clbits)

q0, q1 = qubits
c0, c1 = clbits

circuit.h([q0, q1])
circuit.measure(q0, c0)
circuit.measure(q1, c1)
with circuit.while_loop((clbits, 0b11)):
    circuit.h([q0, q1])
    circuit.measure(q0, c0)
    circuit.measure(q1, c1)

circuit.draw("mpl")

# example output counts: {'01': 334, '10': 368, '00': 322}

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/174a9675-3c8b-4b5e-808e-f7e0f8b9c805-0.avif)

## التعبيرات الكلاسيكية
تحتوي وحدة التعبيرات الكلاسيكية في Qiskit [`qiskit.circuit.classical`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical) على تمثيل استكشافي للعمليات التي تُنفَّذ في وقت التشغيل على القيم الكلاسيكية أثناء تنفيذ الدائرة.

يُظهر المثال التالي كيف يمكنك استخدام حساب التكافؤ لإنشاء حالة GHZ بـ n qubit باستخدام الدوائر الديناميكية. أولاً، أنشئ $n/2$ زوج من أزواج Bell على الـ qubits المتجاورة. ثم ألصق هذه الأزواج ببعضها باستخدام طبقة من بوابات CNOT بين الأزواج. بعد ذلك تقيس الـ qubit الهدف لجميع بوابات CNOT السابقة وتُعيد تعيين كل qubit مقيس إلى الحالة $\vert 0 \rangle$. تطبّق $X$ على كل موقع غير مقيس حيث يكون تكافؤ جميع البتات السابقة فردياً. وأخيراً، تُطبَّق بوابات CNOT على الـ qubits المقيسة لاستعادة التشابك الضائع بسبب القياس.

في حساب التكافؤ، يتضمن العنصر الأول للتعبير المُنشأ رفع كائن Python `mr[0]` إلى عقدة [`Value`](https://docs.quantum.ibm.com/api/qiskit/circuit_classical#value) (يُستخدم `lift` لتحويل الكائنات التعسفية إلى تعبيرات كلاسيكية). هذا غير ضروري لـ `mr[1]` والسجل الكلاسيكي التالي المحتمل، إذ أنهما مدخلات لـ `expr.bit_xor`، ويُجرى أي رفع ضروري تلقائياً في هذه الحالات. يمكن بناء مثل هذه التعبيرات في حلقات وهياكل أخرى.

In [8]:
from qiskit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

num_qubits = 8
if num_qubits % 2 or num_qubits < 4:
    raise ValueError("num_qubits must be an even integer ≥ 4")
meas_qubits = list(range(2, num_qubits, 2))  # qubits to measure and reset

qr = QuantumRegister(num_qubits, "qr")
mr = ClassicalRegister(len(meas_qubits), "m")
qc = QuantumCircuit(qr, mr)

# Create local Bell pairs
qc.reset(qr)
qc.h(qr[::2])
for ctrl in range(0, num_qubits, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Glue neighboring pairs
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

# Measure boundary qubits between pairs,reset to 0
for k, q in enumerate(meas_qubits):
    qc.measure(qr[q], mr[k])
    qc.reset(qr[q])

# Parity-conditioned X corrections
# Each non-measured qubit gets flipped iff the parity (XOR) of all
# preceding measurement bits is 1
for tgt in range(num_qubits):
    if tgt in meas_qubits:  # skip measured qubits
        continue
    # all measurement registers whose physical qubit index < tgt
    left_bits = [k for k, q in enumerate(meas_qubits) if q < tgt]
    if not left_bits:  # skip if list empty
        continue

    # build XOR-parity expression
    parity = expr.lift(
        mr[left_bits[0]]
    )  # lift the first bit to Value so it will be treated like a boolean.
    for k in left_bits[1:]:
        parity = expr.bit_xor(
            mr[k], parity
        )  # calculate parity with all other bits
    with qc.if_test(parity):  # Add X if parity is 1
        qc.x(qr[tgt])

# Re-entangle measured qubits
for ctrl in range(1, num_qubits - 1, 2):
    qc.cx(qr[ctrl], qr[ctrl + 1])

In [9]:
qc.draw(output="mpl", style="iqp", idle_wires=False, fold=-1)

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/d2fdf38a-e874-4de1-9a79-08aab97f9ecc-0.avif)

<span id="store"></span>
### `Store`

يمكنك استخدام تعليمة [`store`](https://docs.quantum.ibm.com/api/qiskit/circuit#store) لحفظ نتيجة تعبير كلاسيكي، إذا كان سيُستخدم هذا التعبير بشكل متكرر. يتم توازي العمليات تلقائيًا، مما يجعل الكود أكثر كفاءة بشكل ملحوظ في وقت التشغيل.

على سبيل المثال، من الأكثر طبيعية والأكثر كفاءة في وقت التشغيل كتابة $B[0] \oplus B[1] \oplus B[2] \ldots$، حيث $B = \neg A$، بدلاً من $(\neg A[0]) \oplus (\neg A[1]) \oplus (\neg A[2]) \ldots$. يحسب التعبير الأول النفي في خطوة موازية واحدة قبل سلسلة XOR، بدلاً من تقييم كل نفي بشكل تسلسلي داخل التعبير.

مثال كامل:

In [10]:
from qiskit.circuit import QuantumCircuit, QuantumRegister, ClassicalRegister
from qiskit.circuit.classical import expr

qregs = QuantumRegister(4, "q")
creg = ClassicalRegister(3, "c")
# temp is a plain ClassicalRegister used as the store target
temp = ClassicalRegister(3, "temp")
qc = QuantumCircuit(qregs, creg, temp)

qc.h([0, 1, 2])
qc.measure([0, 1, 2], creg)

# Store bit-NOT of the full 3-bit register into temp
qc.store(temp, expr.bit_not(creg))

# Compute parity of temp using bit-indexed XOR
parity = expr.bit_xor(
    expr.bit_xor(expr.index(temp, 0), expr.index(temp, 1)),
    expr.index(temp, 2),
)

# Flip q3 if parity of ~creg is 1
with qc.if_test(parity):
    qc.x(3)

qc.measure([0, 1, 2], creg)

qc.draw("mpl")

<Image src="../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/classical-feedforward-and-control-flow/extracted-outputs/f76db731-afa1-4777-9482-25376aa86175-0.avif)

## الخطوات التالية
> **Tip:** - تعلّم كيفية تطبيق فصل ديناميكي دقيق باستخدام [stretch](/guides/stretch).
> - استخدم [تصور جدول الدائرة الزمني](/guides/qiskit-runtime-circuit-timing) لتصحيح الأخطاء وتحسين دوائرك الديناميكية.
> - [نفّذ الدوائر الديناميكية](/guides/execute-dynamic-circuits).